# T1 · Survivability heatmap (CPU, no GPU)

Task **T1** from `Essential_task_delegation/Next_Steps_Roadmap.md`, for **O3**. The survivability heatmap is one of KR3's four required outputs, with the report, the damage map and the counts.

**What it does**
For each of the 645 validation images it builds three maps from R9's predicted mask (8-view TTA):
- **Danger (kernel density):** destroyed pixels weigh 1.0 and damaged pixels 0.5. A Gaussian blur (sigma 25 px) turns them into a local density between 0 and 1.
- **Safety:** the distance to the nearest damaged or destroyed pixel, scaled so that 200 px or more counts as fully safe (1.0).
- **Search priority:** danger × (1 − safety). Dense collapses rank first.

From these it lists, per image:
- the **top 3 hot spots**: centroid, area, and compass position (image top = north);
- the **3 safest open areas**: background pixels farthest from any damage.

`hotspots.json` feeds DisasTeller (T2) and the demo (T3).

**Check against the ground truth:**
- **On damaged or destroyed:** the share of hot spots that overlap ground-truth damaged or destroyed pixels. A perfect mask scores 100%.
- **On destroyed:** the share that overlap ground-truth destroyed pixels. Damaged-only areas also make hot spots, so a perfect mask can't reach 100% here. Each row therefore shows its **ceiling**: the same method run on the ground-truth mask.
- **Local test on the real validation masks:** the ceiling is 46.9% over all images, and 73.3% over the 216 images with destroyed buildings.

**How to run on Kaggle**
1. Settings: CPU only, internet off.
2. Inputs:
   - the DisasterM3 mirror (attached automatically; it supplies the after-images for the overlays);
   - **`d4-r9-val-predictions`**, the dataset made from E1's output. It's found by its `preds/R9_tta` folder.
3. Save & Run All (about 5 minutes). `SMOKE_TEST = True` runs 30 images.
4. Outputs in `/kaggle/working/T1/`:
   - `hotspots.json` (all images), `T1_summary.csv`, `T1_per_image.csv`, `config.json`, `README.md`;
   - `maps/<name>_{kde,safe,priority}.png`: 256 × 256, value × 255;
   - `overlays/<name>.jpg` for 20 images: the 10 xBD and 10 other images with the most ground-truth destroyed pixels.

**Done when:**
- the 20 overlays have been checked by eye;
- the hit rates are reported next to their ceilings.

In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import glob
import json
import re
import time
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
OUT_DIR = Path("/kaggle/working/T1")
SMOKE_TEST = False             # True = 30 images

PRED_SET = "R9_tta"            # E1 exports R9_std and R9_tta; TTA has the best Destroyed IoU
SIGMA_PX = 25                  # danger-map blur (kernel density bandwidth), tuned by eye
DAMAGED_WEIGHT = 0.5           # destroyed pixels weigh 1.0
SAFE_SCALE_PX = 200            # this far from any damage counts as fully safe
HOT_THRESHOLD = 0.05           # search priority above this forms a hot spot
SAFE_MIN_GAP_PX = 150          # the 3 safest areas are at least this far apart
DISPLAY_MAX = 0.5              # colour scale top for danger and priority, the same for every image
N_CHECK = 10                   # overlays for the top-N xBD and top-N other images

hits = sorted(glob.glob(f"/kaggle/input/**/{PRED_SET}", recursive=True))
if not hits:
    raise FileNotFoundError(f"No '{PRED_SET}' folder under /kaggle/input: attach d4-r9-val-predictions (E1's output).")
PRED_DIR = Path(hits[0])
GT_DIR = PRED_DIR.parent.parent / "gt"
names = sorted(p.name for p in PRED_DIR.glob("*.png"))
if SMOKE_TEST:
    names = names[:30]

(OUT_DIR / "maps").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "overlays").mkdir(exist_ok=True)
t_start = time.time()
print(f"✓ T1 survivability heatmap{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
print(f"  Predictions:  {PRED_DIR} ({len(names)} masks)")
print(f"  Ground truth: {GT_DIR}")

✓ T1 survivability heatmap
  Predictions:  /kaggle/input/datasets/abrarmohammedtanzim/d4-r9-val-predictions/E1/preds/R9_tta (645 masks)
  Ground truth: /kaggle/input/datasets/abrarmohammedtanzim/d4-r9-val-predictions/E1/gt


## 1. The three maps, hot spots and safe areas
- **Danger** = GaussianBlur(1.0 × destroyed + 0.5 × damaged, sigma 25 px): the local share of collapsed ground.
- **Safety** = min(distance to the nearest damaged or destroyed pixel / 200 px, 1).
- **Priority** = danger × (1 − safety).
- **Hot spots** are the connected regions where priority ≥ 0.05, ranked by their total priority.
- **Safe areas** are the background pixels farthest from any damage, at least 150 px apart.

All of these are settings in Cell 1.

In [2]:
# ── Cell 2: Maps, hot spots and safe areas ─────────────────────────────────
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(stem):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster' -> ('hurricane_florence', 'xBD'),
    'turkey_earthquake5_post_12' -> ('turkey_earthquake', 'other')."""
    m = XBD_NAME.match(stem)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(stem)
    return re.sub(r"\d+$", "", m.group(1) if m else stem), "other"

def survivability_maps(mask):
    """mask: uint8, 0 background, 1 intact, 2 damaged, 3 destroyed.
    Returns danger, safety and priority (float32, 0-1), and the distance in px to the nearest damage."""
    weight = (mask == 3).astype(np.float32) + DAMAGED_WEIGHT * (mask == 2)
    danger = cv2.GaussianBlur(weight, (0, 0), SIGMA_PX)
    damage = mask >= 2
    dist = (cv2.distanceTransform((~damage).astype(np.uint8), cv2.DIST_L2, 5) if damage.any()
            else np.full(mask.shape, np.inf, np.float32))
    safety = np.minimum(dist / SAFE_SCALE_PX, 1).astype(np.float32)
    return danger, safety, danger * (1 - safety), dist

def compass(x, y, w, h):
    """Where a point sits in the image (top = north): 'north-east', 'west', 'centre', ..."""
    row = ("north", "", "south")[min(int(3 * y / h), 2)]
    col = ("west", "", "east")[min(int(3 * x / w), 2)]
    return "-".join(p for p in (row, col) if p) or "centre"

def hot_spots(priority, k=3):
    """The k strongest connected regions with priority >= HOT_THRESHOLD, ranked by total priority.
    Also returns the label image, so a caller can test each region (its id is in 'label')."""
    n, labels, stats, cents = cv2.connectedComponentsWithStats(
        (priority >= HOT_THRESHOLD).astype(np.uint8), connectivity=8)
    if n <= 1:
        return [], labels
    mass = np.bincount(labels.ravel(), weights=priority.ravel(), minlength=n)
    mass[0] = -1                                    # label 0 is everything below the threshold
    h, w = priority.shape
    spots = []
    for rank, i in enumerate(np.argsort(-mass)[:min(k, n - 1)], start=1):
        x, y = cents[i]
        spots.append({"rank": rank, "label": int(i), "centroid_xy": [int(x), int(y)],
                      "area_px": int(stats[i, cv2.CC_STAT_AREA]),
                      "peak": round(float(priority[labels == i].max()), 3), "where": compass(x, y, w, h)})
    return spots, labels

def safe_areas(dist, mask, k=3):
    """The k open (background) points farthest from any damage, at least SAFE_MIN_GAP_PX apart."""
    if np.isinf(dist).all():
        return []                                   # no damage predicted: the whole image is safe
    score = np.where(mask == 0, dist, -1.0)
    h, w = mask.shape
    yy, xx = np.ogrid[:h, :w]
    areas = []
    for rank in range(1, k + 1):
        y, x = np.unravel_index(np.argmax(score), score.shape)
        if score[y, x] < 0:
            break
        areas.append({"rank": rank, "xy": [int(x), int(y)], "distance_px": int(score[y, x]),
                      "where": compass(x, y, w, h)})
        score[(yy - y) ** 2 + (xx - x) ** 2 < SAFE_MIN_GAP_PX ** 2] = -1
    return areas

# Self-check: one destroyed block -> one hot spot on it, and safe areas far away from it
_m = np.zeros((400, 400), np.uint8)
_m[180:220, 180:220] = 3
_d, _s, _p, _dist = survivability_maps(_m)
_spots, _ = hot_spots(_p)
assert len(_spots) == 1 and _spots[0]["where"] == "centre" and abs(_spots[0]["centroid_xy"][0] - 200) <= 2
_safe = safe_areas(_dist, _m)
assert len(_safe) == 3 and all(a["distance_px"] > 150 for a in _safe)
assert safe_areas(np.full((5, 5), np.inf, np.float32), np.zeros((5, 5), np.uint8)) == []
print("✓ Map functions ready (self-check passed)")

✓ Map functions ready (self-check passed)


In [3]:
# ── Cell 3: Every validation image -> maps, hot spots, safe areas and the ground-truth check ──
def save_map(m, path):
    small = cv2.resize(m, (256, 256), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(path), np.round(np.clip(small, 0, 1) * 255).astype(np.uint8))

records, rows = [], []
for name in tqdm(names, desc="Images"):
    pred = cv2.imread(str(PRED_DIR / name), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(str(GT_DIR / name), cv2.IMREAD_GRAYSCALE)
    stem = Path(name).stem
    event, source = event_and_source(stem)
    danger, safety, priority, dist = survivability_maps(pred)
    spots, labels = hot_spots(priority)
    for s in spots:
        region = labels == s.pop("label")
        s["on_gt_destroyed"] = bool((region & (gt == 3)).any())
        s["on_gt_damage"] = bool((region & (gt >= 2)).any())
    # the same method on the ground-truth mask: the best a perfect segmenter could do
    gt_spots, gt_labels = hot_spots(survivability_maps(gt)[2])
    gt_on_destroyed = sum(bool(((gt_labels == s["label"]) & (gt == 3)).any()) for s in gt_spots)

    records.append({"image": name, "event": event, "source": source,
                    "pred_damaged_px": int((pred == 2).sum()), "pred_destroyed_px": int((pred == 3).sum()),
                    "hot_spots": spots, "safe_areas": safe_areas(dist, pred)})
    rows.append({"image": name, "event": event, "source": source, "gt_destroyed_px": int((gt == 3).sum()),
                 "hot_spots": len(spots), "on_gt_destroyed": sum(s["on_gt_destroyed"] for s in spots),
                 "on_gt_damage": sum(s["on_gt_damage"] for s in spots),
                 "gt_hot_spots": len(gt_spots), "gt_on_destroyed": gt_on_destroyed})
    for kind, m in (("kde", danger), ("safe", safety), ("priority", priority)):
        save_map(m, OUT_DIR / "maps" / f"{stem}_{kind}.png")

per_image = pd.DataFrame(rows)

def hit_rates(df):
    n = max(int(df["hot_spots"].sum()), 1)
    return {"images": len(df), "hot_spots": int(df["hot_spots"].sum()),
            "on_gt_destroyed": round(df["on_gt_destroyed"].sum() / n, 3),
            "ceiling_from_gt_mask": round(df["gt_on_destroyed"].sum() / max(int(df["gt_hot_spots"].sum()), 1), 3),
            "on_gt_damaged_or_destroyed": round(df["on_gt_damage"].sum() / n, 3)}

# Hot spots also come from damaged-only areas, so even a perfect mask can't put all of them on destroyed
# pixels: read on_gt_destroyed against ceiling_from_gt_mask, not against 100%.
summary = pd.DataFrame([
    {"images_from": population, "source": source, **hit_rates(g)}
    for population, df in [("all", per_image), ("with destroyed", per_image[per_image["gt_destroyed_px"] > 0])]
    for source, g in [("all", df), *df.groupby("source")]])
with open(OUT_DIR / "hotspots.json", "w") as f:
    json.dump(records, f, indent=1)
summary.to_csv(OUT_DIR / "T1_summary.csv", index=False)
per_image.to_csv(OUT_DIR / "T1_per_image.csv", index=False)
print(summary.to_string(index=False))
print("\non_gt_destroyed: share of hot spots that touch ground-truth destroyed pixels. ceiling_from_gt_mask: the "
      "same share when the maps come from the ground-truth mask (below 100%, because damaged areas also make "
      "hot spots). on_gt_damaged_or_destroyed: its ceiling is 100%.")

Images:   0%|          | 0/645 [00:00<?, ?it/s]

   images_from source  images  hot_spots  on_gt_destroyed  ceiling_from_gt_mask  on_gt_damaged_or_destroyed
           all    all     645        708            0.345                 0.470                       0.768
           all  other      85         50            0.480                 0.432                       0.600
           all    xBD     560        658            0.334                 0.475                       0.781
with destroyed    all     216        404            0.604                 0.734                       0.886
with destroyed  other      22         33            0.727                 0.704                       0.818
with destroyed    xBD     194        371            0.593                 0.738                       0.892

on_gt_destroyed: share of hot spots that touch ground-truth destroyed pixels. ceiling_from_gt_mask: the same share when the maps come from the ground-truth mask (below 100%, because damaged areas also make hot spots). on_gt_damaged_or_destroye

## 2. Overlays to check by eye
For the 10 xBD and 10 other images with the most ground-truth destroyed pixels, one figure each with four panels: the after-image, then danger, safety (green triangles = the 3 safest open areas) and search priority (numbered cyan circles = the top 3 hot spots). The colour scale is the same for every image (`DISPLAY_MAX`), so images can be compared.

In [4]:
# ── Cell 4: Overlays for the 20 images to check by eye, README and config ──
def find_image(name):
    """The after-image in the mirror (same candidates as the D4 notebooks' resolve_path)."""
    for c in (DATA_ROOT / "train_images" / name, DATA_ROOT / "train_images" / "train_images" / name):
        if c.exists():
            return c
    return None

def overlay(rec):
    name, stem = rec["image"], Path(rec["image"]).stem
    pred = cv2.imread(str(PRED_DIR / name), cv2.IMREAD_GRAYSCALE)
    danger, safety, priority, _ = survivability_maps(pred)
    path = find_image(name)
    image = (cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB) if path
             else np.full((*pred.shape, 3), 128, np.uint8))           # grey if the image is missing
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
    panels = [("After image", None, None, None), ("Danger (kernel density)", danger, "inferno", DISPLAY_MAX),
              ("Safety + 3 safest open areas", safety, "viridis", 1.0),
              ("Search priority + top 3 hot spots", priority, "inferno", DISPLAY_MAX)]
    for ax, (title, m, cmap, vmax) in zip(axes, panels):
        ax.imshow(image)
        if m is not None:
            shown = ax.imshow(m, cmap=cmap, alpha=0.55, vmin=0, vmax=vmax)
            fig.colorbar(shown, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title(title)
        ax.axis("off")
    for a in rec["safe_areas"]:
        axes[2].plot(*a["xy"], "^", ms=12, color="lime", mec="black")
    for s in rec["hot_spots"]:
        x, y = s["centroid_xy"]
        axes[3].plot(x, y, "o", ms=16, mfc="none", mec="cyan", mew=2)
        axes[3].text(x + 14, y, str(s["rank"]), color="cyan", fontsize=13, weight="bold")
    fig.suptitle(f"{stem}  ({rec['event']}, {rec['source']})")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "overlays" / f"{stem}.jpg", dpi=80)
    plt.close(fig)
    return path is not None

check = pd.concat([g.nlargest(N_CHECK, "gt_destroyed_px") for _, g in per_image.groupby("source")])
by_name = {r["image"]: r for r in records}
found = [overlay(by_name[n]) for n in tqdm(check["image"], desc="Overlays")]
if not all(found):
    print(f"⚠ {found.count(False)} after-images not found in the mirror: those overlays have a grey background")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

examples = pd.concat([g.head(3) for _, g in check.groupby("source")])
config = {"task": "T1", "smoke_test": SMOKE_TEST, "predictions": str(PRED_DIR), "images": len(names),
          "sigma_px": SIGMA_PX, "damaged_weight": DAMAGED_WEIGHT, "safe_scale_px": SAFE_SCALE_PX,
          "hot_threshold": HOT_THRESHOLD, "safe_min_gap_px": SAFE_MIN_GAP_PX, "display_max": DISPLAY_MAX,
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# T1 survivability heatmap", "",
    f"{len(names)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}, maps from R9's predicted masks "
    f"(`{PRED_SET}`). Danger = GaussianBlur(1.0 x destroyed + {DAMAGED_WEIGHT} x damaged, sigma {SIGMA_PX} px); "
    f"safety = min(distance to damage / {SAFE_SCALE_PX} px, 1); priority = danger x (1 - safety); "
    f"hot spots = connected regions with priority >= {HOT_THRESHOLD}.", "",
    "## Hot spots against the ground truth", "",
    as_markdown(summary), "",
    "- `on_gt_destroyed`: share of hot spots that touch ground-truth destroyed pixels.",
    "- `ceiling_from_gt_mask`: the same share when the maps come from the ground-truth mask. It is below 100% "
    "because damaged areas also make hot spots, so read `on_gt_destroyed` against it.",
    "- `on_gt_damaged_or_destroyed`: share of hot spots that touch damaged or destroyed pixels (ceiling 100%).", "",
    "## The 20 images to check by eye", "",
    as_markdown(check[["image", "source", "gt_destroyed_px", "hot_spots", "on_gt_destroyed"]]), "",
    "## Examples (3 xBD, 3 other)", "",
    *[f"![{n}](overlays/{Path(n).stem}.jpg)\n" for n in examples["image"]],
    "## Files", "",
    "- `hotspots.json`: per image, the top 3 hot spots (centroid, area, peak, compass position, ground-truth "
    "check) and the 3 safest open areas (point, distance to damage, compass position)",
    "- `maps/<name>_{kde,safe,priority}.png`: 256 x 256, value x 255",
    "- `overlays/<name>.jpg`, `T1_summary.csv`, `T1_per_image.csv`, `config.json`",
]), encoding="utf-8")
print(f"✓ Wrote {len(check)} overlays, README.md and config.json ({(time.time() - t_start) / 60:.1f} min)")

Overlays:   0%|          | 0/20 [00:00<?, ?it/s]

✓ Wrote 20 overlays, README.md and config.json (4.2 min)
